In [ ]:

from google.colab import drive
drive.mount('/content/drive',force_remount=True)

zip_path = "/content/drive/MyDrive/hsi_data_pca.zip"   
!unzip -o "{zip_path}" -d /content/hsi_data
!ls -lh /content/hsi_data/preprocessedd

Mounted at /content/drive
Archive:  /content/drive/MyDrive/hsi_data_pca.zip
   creating: /content/hsi_data/preprocessed/
  inflating: /content/hsi_data/preprocessed/y_train.npy  
  inflating: /content/hsi_data/preprocessed/y_test.npy  
  inflating: /content/hsi_data/preprocessed/X_test.npy  
  inflating: /content/hsi_data/preprocessed/X_train.npy  
   creating: /content/hsi_data/preprocessedd/
  inflating: /content/hsi_data/preprocessedd/y_train.npy  
  inflating: /content/hsi_data/preprocessedd/y_test.npy  
  inflating: /content/hsi_data/preprocessedd/X_test.npy  
  inflating: /content/hsi_data/preprocessedd/X_train.npy  
total 734M
-rw-r--r-- 1 root root 661M Sep  8 10:24 X_test.npy
-rw-r--r-- 1 root root  73M Sep  8 10:24 X_train.npy
-rw-r--r-- 1 root root  73K Sep  8 10:24 y_test.npy
-rw-r--r-- 1 root root 8.1K Sep  8 10:24 y_train.npy


In [ ]:

import os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from tqdm import tqdm

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:

X_train = np.load("/content/hsi_data/preprocessedd/X_train.npy")
y_train = np.load("/content/hsi_data/preprocessedd/y_train.npy")
X_test  = np.load("/content/hsi_data/preprocessedd/X_test.npy")
y_test  = np.load("/content/hsi_data/preprocessedd/y_test.npy")

print("X_train", X_train.shape, "y_train", y_train.shape)
print("X_test", X_test.shape, "y_test", y_test.shape)

X_train (1018, 25, 25, 30) y_train (1018,)
X_test (9231, 25, 25, 30) y_test (9231,)


In [ ]:

class HSIDataset(Dataset):
    def __init__(self, X, y, augment=False):
        
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]  
        
        if self.augment:
            if random.random() > 0.5:
                x = np.flip(x, axis=0).copy()  
            if random.random() > 0.5:
                x = np.flip(x, axis=1).copy()  
            k = random.randint(0,3)
            if k:
                x = np.rot90(x, k, axes=(0,1)).copy()
        
        x = np.ascontiguousarray(x.transpose(2,0,1), dtype=np.float32)
        return torch.from_numpy(x), torch.tensor(self.y[idx], dtype=torch.long)

batch_size = 64   
train_ds = HSIDataset(X_train, y_train, augment=True)
test_ds  = HSIDataset(X_test, y_test, augment=False)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [6]:
# Cell C5
class HybridSN(nn.Module):
    def __init__(self, in_bands, num_classes):
        """
        in_bands: number of spectral bands after PCA (e.g., 30)
        Input patches expected as (batch, C=in_bands, H=25, W=25)
        We'll adapt to 3D conv which expects (N, 1, D, H, W) where D=in_bands
        """
        super().__init__()
        # 3D convolutional block (spectral-spatial)
        # We use padding to preserve spatial dims; spectral dimension shrinks depending on kernel
        self.conv3d_1 = nn.Conv3d(1, 8, kernel_size=(7,3,3), padding=(3,1,1))
        self.bn3d_1   = nn.BatchNorm3d(8)
        self.conv3d_2 = nn.Conv3d(8,16,kernel_size=(5,3,3), padding=(2,1,1))
        self.bn3d_2   = nn.BatchNorm3d(16)
        self.conv3d_3 = nn.Conv3d(16,32,kernel_size=(3,3,3), padding=(1,1,1))
        self.bn3d_3   = nn.BatchNorm3d(32)

        # After 3D convs, we will have shape (N, 32, D, H, W)
        # We'll collapse (channel * depth) -> channels for 2D conv
        # 2D conv block (spatial refinement)
        # in_channels_2d = 32 * in_bands (depth)  -> this can be large; instead, we compress depth via pointwise conv
        # So first reduce spectral-depth via a 1x1 conv2d after reshaping
        # Implementation approach:
        # 1) reshape to (N, 32*D, H, W) then apply a 2D conv to mix those features.
        self.conv2d_1 = nn.Conv2d(32 * in_bands, 64, kernel_size=3, padding=1)
        self.bn2d_1   = nn.BatchNorm2d(64)
        self.conv2d_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2d_2   = nn.BatchNorm2d(64)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

        # Classifier
        self.fc1 = nn.Linear(64, 256)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 128)
        self.drop2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape: (N, C=in_bands, H, W)
        N, C, H, W = x.shape
        # convert to (N, 1, D, H, W)
        x = x.unsqueeze(1)  # (N,1,C,H,W) where C==D
        x = F.relu(self.bn3d_1(self.conv3d_1(x)))
        x = F.relu(self.bn3d_2(self.conv3d_2(x)))
        x = F.relu(self.bn3d_3(self.conv3d_3(x)))  # (N,32,D,H,W)
        # reshape -> (N, 32*D, H, W)
        N, ch, D, H, W = x.shape
        x = x.permute(0,1,2,3,4).contiguous().view(N, ch*D, H, W)
        x = F.relu(self.bn2d_1(self.conv2d_1(x)))
        x = F.relu(self.bn2d_2(self.conv2d_2(x)))
        x = self.avgpool(x).view(N, -1)  # (N, 64)
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = self.fc3(x)
        return x

# instantiate
in_bands = X_train.shape[-1]  # 30
num_classes = int(y_train.max() + 1)
model = HybridSN(in_bands=in_bands, num_classes=num_classes).to(device)
print(model)

HybridSN(
  (conv3d_1): Conv3d(1, 8, kernel_size=(7, 3, 3), stride=(1, 1, 1), padding=(3, 1, 1))
  (bn3d_1): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3d_2): Conv3d(8, 16, kernel_size=(5, 3, 3), stride=(1, 1, 1), padding=(2, 1, 1))
  (bn3d_2): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3d_3): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (bn3d_3): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2d_1): Conv2d(960, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2d_1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2d_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2d_2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc1): Linear(in_features=64, out_features=256, bias=

In [ ]:

import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:



num_epochs = 200
best_val_acc = 0.0
patience = 20
patience_counter = 0

for epoch in range(1, num_epochs+1):
    model.train()
    train_loss = 0.0

    
    for Xb, yb in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * Xb.size(0)

    train_loss /= len(train_ds)

    
    model.eval()
    val_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            outputs = model(Xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item() * Xb.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(yb.cpu().numpy())

    val_loss /= len(test_ds)
    val_acc = accuracy_score(y_true, y_pred)

    print(f"Epoch {epoch} | TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | ValAcc={val_acc:.4f}")

    
    scheduler.step(val_loss)

    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc
        }, os.path.join(checkpoint_dir, "HybridSN_best_full.pth"))
        print("Saved best full checkpoint.")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Train Epoch 1: 100%|██████████| 16/16 [00:04<00:00,  3.52it/s]


Epoch 1 | TrainLoss=2.2745 | ValLoss=1.8423 | ValAcc=0.3448
Saved best full checkpoint.


Train Epoch 2: 100%|██████████| 16/16 [00:02<00:00,  5.48it/s]


Epoch 2 | TrainLoss=1.5011 | ValLoss=1.1906 | ValAcc=0.6225
Saved best full checkpoint.


Train Epoch 3: 100%|██████████| 16/16 [00:03<00:00,  5.33it/s]


...

Epoch 138 | TrainLoss=0.0058 | ValLoss=0.0537 | ValAcc=0.9912


Train Epoch 139: 100%|██████████| 16/16 [00:03<00:00,  5.15it/s]


Epoch 139 | TrainLoss=0.0094 | ValLoss=0.0535 | ValAcc=0.9910


Train Epoch 140: 100%|██████████| 16/16 [00:03<00:00,  5.24it/s]


Epoch 140 | TrainLoss=0.0064 | ValLoss=0.0544 | ValAcc=0.9907
Early stopping triggered.


In [ ]:




import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


checkpoint_path = os.path.join(checkpoint_dir, "HybridSN_best_full.pth")
checkpoint = torch.load(checkpoint_path, map_location=device)

 
print("Checkpoint conv3d_1.weight shape:", checkpoint['model_state_dict']['conv3d_1.weight'].shape)

 
 
in_bands = 30
num_classes = int(y_train.max() + 1)

model = HybridSN(in_bands=in_bands, num_classes=num_classes).to(device)

 
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Loaded checkpoint successfully. Best val_acc:", checkpoint['val_acc'])

 
y_true, y_pred = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        outputs = model(Xb)
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(yb.cpu().numpy())

 
val_acc = accuracy_score(y_true, y_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

 
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Checkpoint conv3d_1.weight shape: torch.Size([8, 1, 7, 3, 3])
Loaded checkpoint successfully. Best val_acc: 0.9912252193695158
Validation Accuracy: 0.9912

Confusion Matrix:
[[  42    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [   0 1268    0    0    0    0    0    0    0    6    6    0    0    1
     5    0]
 [   0    0  743    2    0    0    0    0    0    0    0    1    1    0
     0    0]
 [   0    0    0  214    0    0    0    0    0    0    0    0    0    0
     0    0]
 [   0    0   12    0  416    0    0    0    0    6    0    0    1    0
     0    0]
 [   0    9    0    0    0  647    0    0    0    0    1    0    0    0
     0    0]
 [   0    0    0    0    0    0   26    0    0    0    0    0    0    0
     0    0]
 [   0    0    0    0    0    0    0  431    0    0    0    0    0    0
     0    0]
 [   0    0    0    0    0    0    0    0   18    0    0    0    0    0
     0    0]
 [   0    0    0    0    0    0    0    0    0  868    5    2

In [ ]:
import os
import torch

 
checkpoint_dir = "./HybridSN_checkpoints"   
os.makedirs(checkpoint_dir, exist_ok=True)

 
checkpoint_path = os.path.join(checkpoint_dir, "HybridSN_full_checkpoint.pth")

 
torch.save({
    'epoch': epoch,                      
    'model_state_dict': model.state_dict(),    
    'optimizer_state_dict': optimizer.state_dict(),   
    'scheduler_state_dict': scheduler.state_dict(),   
    'val_acc': best_val_acc              
}, checkpoint_path)

print(f"Checkpoint saved successfully at {checkpoint_path}")

Checkpoint saved successfully at ./HybridSN_checkpoints/HybridSN_full_checkpoint.pth


In [ ]:
from google.colab import files
import os

 
checkpoint_path = './HybridSN_checkpoints/HybridSN_full_checkpoint.pth'

 
if os.path.exists(checkpoint_path):
    files.download(checkpoint_path)
else:
    print("Checkpoint file not found!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>